In [1]:
from shared_lib.local import LOCAL_ENV, LOCAL_RUN
from shared_lib.spark import (
    get_spark_session,
)
spark = get_spark_session(
    app_name="Aggtrades Ingestion Job", master=True, jars=True, local_run=LOCAL_RUN, minio=LOCAL_ENV
)

26/04/13 17:51:14 WARN Utils: Your hostname, Nguyens-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.13 instead (on interface en0)
26/04/13 17:51:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/anhtu/.ivy2/cache
The jars for the packages stored in: /Users/anhtu/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-69ab9021-bc95-4028-b176-ebf1a4aef931;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/anhtu/.pyenv/versions/3.11.11/envs/spark/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.7.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 111ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.7.1 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------------------

In [2]:
from pyspark.sql import types
schema = types.StructType(
    [
        types.StructField("agg_trade_id", types.LongType(), True),
        types.StructField("price", types.DoubleType(), True),
        types.StructField("quantity", types.DoubleType(), True),
        types.StructField("first_trade_id", types.LongType(), True),
        types.StructField("last_trade_id", types.LongType(), True),
        types.StructField("timestamp", types.LongType(), True),
        types.StructField("is_buyer_maker", types.BooleanType(), True),
        types.StructField("is_best_match", types.BooleanType(), True),
    ]
)

In [3]:
import os
from pyspark.sql import functions as F
data_lake_bucket = os.getenv("DATA_LAKE_BUCKET", "binance-data-lake")
date = "2025-09-29"
event_date = date
base_path = f"s3a://{data_lake_bucket}/raw_zone/aggtrades/"
read_path = f"{base_path}/date={date}/"

df = spark.read \
    .option("header", "false") \
    .option("basePath", base_path) \
    .schema(schema) \
    .csv(read_path) \
    .withColumn(
        "ingested_at", F.to_timestamp(F.col("ingestion_ts"), "yyyyMMdd_HHmmss'Z'")
    ).withColumn(
        "event_time", (F.col("timestamp") / 1_000_000).cast("timestamp")
    ).withColumn("event_date", F.col("event_time").cast("date")).withColumn(
        "created_at", F.current_timestamp()
    ).drop("ingestion_ts").drop(
        "date"
    ).filter(F.col("event_date") == event_date)

26/04/13 17:51:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
df.createOrReplaceTempView("aggtrades_raw")

In [5]:
spark.sql("""
    select agg_trade_id, symbol, max(ingested_at) as latest_ingested_at
    from aggtrades_raw
    group by agg_trade_id, symbol
    """).createOrReplaceTempView("aggtrades_deduped")

In [6]:
df_deduped = spark.sql("""
    select r.*
    from aggtrades_raw r
    join aggtrades_deduped d on r.agg_trade_id = d.agg_trade_id and r.symbol = d.symbol and r.ingested_at = d.latest_ingested_at
    """)

In [7]:
write_path = f"s3a://{data_lake_bucket}/landing_zone/aggtrades/"
df_deduped.repartition("symbol").write.mode("overwrite").partitionBy(
    "event_date", "symbol"
).parquet(write_path)

In [8]:
read_path = f"s3a://{data_lake_bucket}/landing_zone/aggtrades"
spark.read.parquet(read_path).createOrReplaceTempView("aggtrades_landing")

In [9]:
spark.sql("""
    select symbol, event_date, count(*) as num_records
    from aggtrades_landing
    group by symbol, event_date
    order by event_date desc, symbol
    """).show(truncate=False)

+-------+----------+-----------+
|symbol |event_date|num_records|
+-------+----------+-----------+
|ADAUSDT|2025-09-29|70081      |
|BTCUSDT|2025-09-29|794436     |
|DOTUSDT|2025-09-29|32521      |
|ADAUSDT|2025-09-28|51249      |
|BTCUSDT|2025-09-28|521551     |
|DOTUSDT|2025-09-28|27033      |
|ADAUSDT|2025-09-27|38243      |
|BTCUSDT|2025-09-27|351179     |
|DOTUSDT|2025-09-27|17711      |
+-------+----------+-----------+



26/04/13 18:09:27 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /private/var/folders/21/8r4f38z90tz5zgftyt7rck640000gn/T/blockmgr-b4ff8839-e50a-4f73-90b6-3d266222237b. Falling back to Java IO way
java.io.IOException: Failed to delete: /private/var/folders/21/8r4f38z90tz5zgftyt7rck640000gn/T/blockmgr-b4ff8839-e50a-4f73-90b6-3d266222237b
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBl